In [ ]:
"""
<분석 과정>
 1. 데이터 수집
 2. 데이터 전처리(확인 후 이상 없으면 바로 진행)
   2.1. 데이터 가공
 3. 가공된 데이터 특성간의 상관관계 확인
    -> 선형관계 또는 수평관계의 산점도 확인
       --> 선형관계 분포 : 회귀분석 진행
       --> 수평관계 분포 : 분류분석 진행
    -> 상관계수 확인(상관성이 있는지 확인)
 4. 특성들의 유의미성(p-value) 검증(또는 상관관계 검증이라고 칭합니다.)
 5. 상관관계 검증에 따른 데이터 후처리(가공)
   5.1 독립변수와 종속변수로 가공
 6. 훈련하기
     - 훈련을 위한 데이터 전처리하기 : 데이터 분류하기
                                   : 훈련vs테스트
                                   : 훈련vs검증vs테스트
     - 훈련모델 생성
     - 훈련 시키기
     - 훈련 및 테스트(검증) 정확도(또는 결정계수) 확인
       (과대 또는 과소적합 여부 확인)
       -- 과대 또는 과소적합인 경우 튜닝 진행
          -- 튜닝(정규화, 특성늘리기 등등...전처리 수행)
     - 예측하기
     - 오차 확인하기(평가하기)
       -- 평균절대오차(MAE)
       -- 평균제곱오차(MSE)
       -- 결정계수(R2)
       --> 모델 성능은 "결정계수"가 높고, 오차가 낮은값을 선택
           (단, 전제조건은 일반화 되었다는 가정하에~)
           
 7. 훈련 시 사용된 특성들 중 훈련에 중요하게 사용된 특성들 확인
    --> 특성중요도라고 칭합니다.
 8. 중요한 특성들을 기준으로 5번부터 다시 수행해 보기
 9. 하이퍼파라메터 튜닝 (정확도-결정계수 성능 올리기)
    --> 튜닝 클래스 또는 함수 사용
 10. 각 분석 방법에 속한 전체 모델 적용
 11. 최종 모델 선정
    - 6번의 오차(MAE, MSE, R2)를 모델별로 비교하여 선정
 12. 선정된 모델로 최종 테스트데이터로 예측 및 평가하기
 13. 선정된 "모델 저장"하기
 14. 서비스하고자하는 외부 프로그램에서 저장된 모델 불러들여서 사용하면 됨...
     - 외부 서비스하는 프로그램에서는 예측에 사용될 독립변수만 받아들이고,
     - 예측만 수행하면 됩니다.
     - 절대로, 서비스하는 곳에서는 모델을 훈련하거나 평가하면 안됩니다.
     - 웹(Web) 서비스를 하려면, Back-End 영역에 모델을 복사해서 넣은 후 사용
"""

### 주택가격 예측하기

In [ ]:
"""
 - 분석 주제 : 회귀모델을 사용한 주택가격 예측
 - 사용 데이터셋 : 01_회귀_주택가격예측_데이터셋_Row_1000.csv
"""

### 사용할 라이브러리 불러들이기

In [ ]:
### 행렬 데이터 처리 및 조작
import pandas as pd

### 시각화 처리
# 기본 시각화 라이브러리
import matplotlib.pyplot as plt

# 통계적 시각화 라이브러리(보통 기본 시각화보다 향상된 라이브러라고 생각하면 됨)
import seaborn as sns

# 시각화 시 그래프의 너비/높이를 기본 설정하기(단위=인치)
plt.rcParams["figure.figsize"] = (12, 5)

# 시각화시 한글 깨짐 처리를 위한 폰트 지정(Windows OS 기준)
plt.rcParams["font.family"] = "Malgun Gothic"

# 시각화시 마이너스(-) 기호 깨짐 방지 처리
plt.rcParams["axes.unicode_minus"] = False

### 상관관계 검정 라이브러리(p-value)
from scipy.stats import spearmanr, pearsonr

### 훈련:검증:테스트 데이터로 분리하는 라이브러리
from sklearn.model_selection import train_test_split

### 데이터스케일링(표준화 라이브러리)
from sklearn.preprocessing import StandardScaler

### 회귀모델 #####################
# 선형회귀모델 (선형, 다항, 다중)
from sklearn.linear_model import LinearRegression

# 릿지, 라쏘
from sklearn.linear_model import Ridge, Lasso

# 앙상블 Regressor 모델 - 랜덤포레스트
from sklearn.ensemble import RandomForestRegressor
# 앙상블 Regressor 모델 - 엑스트라트리
from sklearn.ensemble import ExtraTreesRegressor
# 앙상블 Regressor 모델 - 그레디언트부스트
from sklearn.ensemble import GradientBoostingRegressor
# 앙상블 Regressor 모델 - 히스트그레디언트부스트
from sklearn.ensemble import HistGradientBoostingRegressor

# 앙상블 Regressor 모델 - 엑스지부스트
from xgboost import XGBRegressor

### 평가에 사용할 라이브러리
# 평균절대오차, 평균제곱오차, 결정계수(score 함수 결과와 동일)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### 데이터 불러들이기

In [ ]:
### 불러들인 데이터 저장할 변수명 : df_org
file_path = "../data/housing_price_1000.csv"
df_org = pd.read_csv(file_path)

display(df_org.info())
display(df_org.describe())
df_org.head(5)

### 이상치 여부를 시각화(박스플롯)으로 확인해보기

In [ ]:
plt.figure(figsize=(10, 5))
plt.title("주택가격 예측 데이터 이상치 확인")

df_org.boxplot()

plt.show()

### 상관관계 확인하기

In [ ]:
### 특성들 간의 관계 확인
# - 종속변수를 기준으로 독립변수들과 비교 확인
#   (독립변수 : 면적, 층수, 역까지_도보시간, 주변_편의시설_점수)
#   (종속변수 : 주택_가격)
# - 선형성(회귀분석)을 띄는지 비선형선(분류분석)을 띄는지 확인
# - 상관관계표 및 상관관계도(시각화)로 표현

##### 상관관계표로 확인

In [ ]:
correlation_metrix = df_org.corr()
correlation_metrix

##### 상관관계도(시각화)로 확인하기

In [ ]:
plt.figure(figsize=(10, 8))
plt.title("상관관계 매트릭스 히트맵 그리기")

sns.heatmap(correlation_metrix, annot=True, cmap="coolwarm", fmt=".3f", linewidths=0.5)

plt.show()

### 산점행렬도 : 특성들간의 관계를 산점도(선형/비선형 분포) 확인하기

In [ ]:
### 산점행렬도 시각화 함수
sns.pairplot(df_org)

### 제목(pairplot 함수 사용시)
plt.suptitle("주택가격 예측 데이터셋 상관관계 분포 확인 - 산점행렬도", y=1.02)

plt.show()

### 상관관계 검증하기(종속변수를 기준으로 독립변수들 간의 유의미성 확인)

In [ ]:
### 유의미성 : p-value < 0.05
### 독립변수와 종속변수로 분리해야함
# - 독립변수 명 : X(대문자)
# - 종속변수 명 : y

X = df_org.iloc[:, :-1]
y = df_org["주택_가격"]

X.shape, y.shape

##### 상관관계 검정 하기 (스피어만 방식)

In [ ]:
### 샘플로 독립변수 "면적"과 종속변수와의 유의미성 확인
s, p = spearmanr(X["면적"], y)
print(f"상관계수={s:.5f} , p-value={p:.5f}")

if p < 0.05 :
    print("p-value < 0.05 이므로 [유의미 함]")
else :
    print("p-value >= 0.05 이므로  [유의미 하지 않음]")


In [ ]:
### 모든 독립변수와 종속변수 간의 유의미성(p-value)를 각각 확인하기
# - 각 독립변수와 종속변수의 유의미성 확인 후 
#   "유의미 함" 또는 "유의미 하지 않음"을 출력해 주세요..

for col_name in X.columns :
    # print(col_name)
    s, p = spearmanr(X[col_name], y)
    print(f"\n 독립변수[{col_name}]와 종속변수[주택_가격] : 상관계수={s:.5f} , p-value={p:.5f}")

    if p < 0.05 :
        print("p-value < 0.05 이므로 [유의미 함]")
    else :
        print("p-value >= 0.05 이므로  [유의미 하지 않음]")

##### 피어스만으로도 유의미성 확인

In [ ]:
for col_name in X.columns :
    # print(col_name)
    s, p = pearsonr(X[col_name], y)
    print(f"\n 독립변수[{col_name}]와 종속변수[주택_가격] : 상관계수={s:.5f} , p-value={p:.5f}")

    if p < 0.05 :
        print("p-value < 0.05 이므로 [유의미 함]")
    else :
        print("p-value >= 0.05 이므로  [유의미 하지 않음]")

### 모델 훈련을 위한 데이터 분리하기 (훈련:검증:테스트=6:2:2로 분리)

In [ ]:
"""
사용 변수명 : X_train, y_train, X_val, y_val, X_test, y_test
"""
### 먼저 [훈련 : (검증+테스트) = 60% : 40%]로 분리
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

### 나머지 [검증 : 테스트 = 50% : 50%]로 분리
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

### 데이터스케일링(표준화) 처리하기(훈련, 검증, 테스트 모두 처리)

In [ ]:
### 변수명 : X_train_scaled, X_val_scaled, X_test_scaled
# 1. 클래스 생성
ss = StandardScaler()

# 2. 훈련 독립변수로 패턴 찾기
ss.fit(X_train)

# 3. 훈련,검증,테스트 독립변수 변환하기
X_train_scaled = ss.transform(X_train)
X_val_scaled   = ss.transform(X_val)
X_test_scaled  = ss.transform(X_test)

"""<데이터 활용 순서 = 훈련 순서와 동일>
 1. 훈련(학습)용 데이터 : 훈련데이터만 사용
 2. 과적합 여부 확인용 데이터 : 훈련 및 검증 데이터만 사용
   : 일반화 될때까지 확인하는 데이터로 훈련 및 검증 데이터만 사용됨
 3. 모델 선별을 위해서는 : 훈련/검증 데이터 모두에 대한 정확도 및 오차 확인이 필요함
 4. 최종 예측 및 오차 확인 용 : 테스트 데이터만 사용
"""
print(X_train_scaled.shape, y_train.shape)
print(X_val_scaled.shape, y_val.shape)
print(X_test_scaled.shape, y_test.shape)

### 회귀모델 훈련하기

In [ ]:
"""
여러 모델을 훈련시키기 위해서는 반복을 통해서 진행.
 - 같은 코드가 반복되면 코드량이 많아짐
 - 반복 전에 하나의 모델로 샘플 코드를 작성 후 반복 진행
"""

##### 선형회귀모델 하나만 우선 훈련시키는 코드 샘플 만들기

In [ ]:
"""
1. 선형회귀모델로 훈련시키기
2. 훈련/검증/테스트 데이터 모두 정확도(결정계수), 예측, 오차(MAE, MSE, R2-score) 확인까지
   - 과적합여부는 [훈련-검증]으로 확인
"""
# 1. 모델 클래스 생성하기
lr = LinearRegression()
print(f"----------------- 클래스(모델) 이름 : {lr.__class__.__name__} -----------------")

# 2. 모델 훈련(학습=공부) 시키기 : 패턴 찾기(기울기=가중치 및 절편 찾기)
lr.fit(X_train_scaled, y_train)

# 3. 정확도(회귀=결정계수) 확인하기(훈련이 잘 되었는지와 과적합 여부 확인)
#  - 훈련 > 검증 > 테스트 or 훈련 > 검증 < 테스트 or 훈련 > 검증 = 테스트
train_score = lr.score(X_train_scaled, y_train)
val_score   = lr.score(X_val_scaled, y_val)
test_score  = lr.score(X_test_scaled, y_test)
print(f"{train_score:.4f}, {val_score:.4f}, {test_score:.4f}, {train_score-val_score:.4f}")

# 4. 예측하기(오차 확인에 사용)
train_pred = lr.predict(X_train_scaled)
val_pred   = lr.predict(X_val_scaled)
test_pred  = lr.predict(X_test_scaled)

# 5. 오차 확인(향후 여러 모델들 중 선별에 사용)
### MAE 확인
train_mae = mean_absolute_error(y_train, train_pred)
val_mae   = mean_absolute_error(y_val,   val_pred)
test_mae  = mean_absolute_error(y_test,  test_pred)
print(f"MAE : {train_mae:.4f}, {val_mae:.4f}, {test_mae:.4f}")

### MSE 확인
train_mse = mean_squared_error(y_train, train_pred)
val_mse   = mean_squared_error(y_val,   val_pred)
test_mse  = mean_squared_error(y_test,  test_pred)
print(f"MSE : {train_mse:.4f}, {val_mse:.4f}, {test_mse:.4f}")

### R2-score(정확도=결정계수=위에서 score 함수 결과와 동일) 확인
train_r2 = r2_score(y_train, train_pred)
val_r2   = r2_score(y_val,   val_pred)
test_r2  = r2_score(y_test,  test_pred)
print(f"R2-score : {train_r2:.4f}, {val_r2:.4f}, {test_r2:.4f}")
print(f"과적합여부 : 훈련[{train_r2:.4f}] - 검증[{val_r2:.4f}] = {train_r2 - val_r2:.4f}")

### 8개 회귀모델 모두 훈련 수행하기

In [ ]:
"""
8개 회귀모델 모두 훈련 시키기
 - 결과는 아래와 같이 출력하기
   -> 모델명, MAE(훈련, 검증, 테스트), MSE(훈련, 검증, 테스트), R2(훈련, 검증, 테스트), 과적합여부(훈련-검증)
      (소수점은 4째자리 까지...)

 - 최종 모델 선정까지 해주세요..
   <모델 선정 우선 순위>
    1. 훈련 결정계수가 1이 아니면서, 과소적합이 일어나지 안은 모델들 중에..
    2. 훈련 결정계수가 높은 그룹 추출
    3. 2번 중에 일반화가 잘된 그룹 추출
    4. 3번 중에 훈련 MAE 오차가 작은 그룹 추출 
    5. 4번 중에 검증 MAE 오차가 작은 그룹 추출
    6. 5번 중에 테스트 MAE 오차가 작은 그룹 추출

    (보통 4번~5번 단계에서 선정이 되는 경우가 많음)
    (추출된 상위 그룹 중에 최종 선택은 분석가 재량 -> 선호도에 따름)
"""

In [ ]:
### 반복을 위해서 8개 모델을 리스트에 담기
lr = LinearRegression()
ridge = Ridge()
lasso = Lasso()
rf = RandomForestRegressor()
et = ExtraTreesRegressor()
gb = GradientBoostingRegressor()
hgb = HistGradientBoostingRegressor()
xgb = XGBRegressor()

models = [lr, ridge, lasso, rf, et, gb, hgb, xgb]

for model in models :    
    print(f"--------------모델명[{model.__class__.__name__}]----------")

    # 2. 모델 훈련(학습=공부) 시키기 : 패턴 찾기(기울기=가중치 및 절편 찾기)
    model.fit(X_train_scaled, y_train)

    # 3. 정확도(회귀=결정계수) 확인하기(훈련이 잘 되었는지와 과적합 여부 확인)
    #  - 훈련 > 검증 > 테스트 or 훈련 > 검증 < 테스트 or 훈련 > 검증 = 테스트
    train_score = model.score(X_train_scaled, y_train)
    val_score   = model.score(X_val_scaled, y_val)
    test_score  = model.score(X_test_scaled, y_test)
    print(f"{train_score:.4f}, {val_score:.4f}, {test_score:.4f}, {train_score-val_score:.4f}")

    # 4. 예측하기(오차 확인에 사용)
    train_pred = model.predict(X_train_scaled)
    val_pred   = model.predict(X_val_scaled)
    test_pred  = model.predict(X_test_scaled)

    # 5. 오차 확인(향후 여러 모델들 중 선별에 사용)
    ### MAE 확인
    train_mae = mean_absolute_error(y_train, train_pred)
    val_mae   = mean_absolute_error(y_val,   val_pred)
    test_mae  = mean_absolute_error(y_test,  test_pred)
    print(f"MAE : {train_mae:.4f}, {val_mae:.4f}, {test_mae:.4f}")

    ### MSE 확인
    train_mse = mean_squared_error(y_train, train_pred)
    val_mse   = mean_squared_error(y_val,   val_pred)
    test_mse  = mean_squared_error(y_test,  test_pred)
    print(f"MSE : {train_mse:.4f}, {val_mse:.4f}, {test_mse:.4f}")

    ### R2-score(정확도=결정계수=위에서 score 함수 결과와 동일) 확인
    train_r2 = r2_score(y_train, train_pred)
    val_r2   = r2_score(y_val,   val_pred)
    test_r2  = r2_score(y_test,  test_pred)
    print(f"R2-score : {train_r2:.4f}, {val_r2:.4f}, {test_r2:.4f}")
    print(f"과적합여부 : 훈련[{train_r2:.4f}] - 검증[{val_r2:.4f}] = {train_r2 - val_r2:.4f}")

In [ ]:
"""
<해석>
 1. 과대 및 과소 적합이 발생한 모델은 제외 시킴
 2. 정확도가 높은 그룹은 RF, GB, HGB 모델
 3. 일반화가 잘된 그룹은 GB, HGB 모델
 4. 오차가 낮은 그룹은 HGB모델
 따라서, 정확도가 가장 높고, 오차가 가장 낮으면서 일반화가 잘된 HGB 모델로 선정하는 것이 타당함
 단, 과적합이 발생한 모델들은 일반화를 위한 처리 이후 다시 확인할 필요성은 있음
"""

### 모델 성능 향상을 위한 하이퍼파라메터 튜닝하기

In [ ]:
"""
<하이퍼파라메터 튜닝>
 - 디폴트 파라메터로 진행한 모델들의 결과값을 "기준"으로...
 - 좀 더 성능을 높일 수 있는 방법을 찾는 과정
 - 하이퍼파라메터는 사람이 개입하여 값들의 범위를 지정하는 방식임
 - 지정된 범위의 하이퍼파라메터를 이용하여, 자동으로 찾아주는 클래스 사용
 
<하이퍼파라메터 튜닝 모델>
 - 모델(클래스) : GridSearchCV()
 - 패키지 : sklearn.model_selection
 
<GridSearchCV()>
 - 각 모델에 대한 하이퍼파라메터의 속성 값들을 스스로 찾아서 훈련까지 수행합니다.
 - "훈련"이라 함은 모델 훈련을 의미합니다.
 - 훈련이 완료된 최적의 모델을 반환 받아서 예측을 수행하게 됩니다.
 - 내부적으로 교차검증(CV, Cross Validation)을 수행하면서 가장 최적의
   하이퍼파라메터 값들을 찾아내는 방법을 사용합니다.
   
<교차검증(CV, Cross Validation)>
 - 데이터를 자체적으로 독립변수와 종속변수의 여러 그룹으로 쪼개어(Fold)
 - 교차하면서(그룹의 값들을 섞어 가면서) 검증하는 단계
 - 이때, 검증은 하이퍼파라메터의 범위 값들을 기준으로 각 훈련결과를 비교하면서 검증
 
<시스템 성능 고려해야함>
 - 여러번에 걸친 교차검증이 진행되기에, 시스템 성능에 따라 느리거나 빠르게 결과를 도출
 - 시스템 자원인 CPU Core(코어)는 가급적 전체 사용하게 됩니다.(속도 측면에서)
 
<하이퍼파라메터 튜닝을 하지 않아도 되는 모델>
 - 선형회귀모델은 튜닝 속성이 없습니다.
   (선형, 다항모델은 튜닝 안함, 다중모델의 경우에는 특성공학 직접 적용)
 - 릿시, 라쏘 모델은 강도에 대한 튜닝만 진행되기에 보통 직접 진행합니다.
 - 앙상블 모델들이 자동 튜닝 대상이 됩니다.
"""

### 하이퍼파라메터 튜닝 클래스로 랜덤포래스트 샘플링하기

In [ ]:
"""
<앙상블모델 배깅(Bagging)과 부스팅(Boosting) 방식>
 - 앙상블 모델 각각은 자체적으로 여러개의 모델(트리 모델-의사결정나무)로 구성되어 있음
 - 훈련데이터를 여러 모델에 적용하는 방식에 따라 배깅과 부스팅 방식으로 구분함

<배깅>
 - 사용되는 여러개의 모델은 서로간에 독립적으로 사용됨(연관성 없음)
 - 훈련데이터는 여러 모델에 랜덤하게 적용되며, 중복되는 데이터가 있을 수도 있음
 - 훈련데이터를 사용 후 반환하는 방식 > 반환 후 다시 랜덤하게 추출하여 사용됨
 - 대표적인 앙상블 모델은 랜덤포레스트 모델이 있음
 - 모델 : 램덤포레스트, 엑스트라트리
 - 예측 : 회귀에서는 평균값을, 분류에서는 과반수에 따라 경정됨
 ** 과적합을 해소하는 모델로 주로 사용됨
 
<부스팅>
 - 사용되는 여러개의 모델은 연관성을 가짐
 - 사용된 이전모델의 오차를 보완하는 방식으로 사용됨
 - 데이터는 랜덤하게 추출되어 사용되지만,
   이전 모델훈련에서 오차가 크게 발생했던 훈련데이터에 가중치를 부여하는 방식으로
   데이터가 모델에 적용됨
 - 대표적인 앙상블 모델은 그레디언트 부스팅 모델이 있음
 - 모델 : 그레디언트 부스팅, 히스토그램 그레디언트 부스팅
 ** 정확도를 높이는 모델로 주로 사용됨
 
<실제 모델 적용시>
 - 앙상블 모델이 아닌 단일모델(결정트리)을 우선 사용
 - 과적합 또는 정확도 문제 발생 시 배깅 또는 부스팅 방식으로 확대 적용
 - 앙상블 모델은 랜덤포레스트를 처음 적용하는 것이 일반적임
 - 최종 모델 선정 시에는 모든 앙상블 모델을 훈련시킨 후 과적합/정확도 등을 고려하여
   선정하게 됨
"""

### 하이퍼파라메터 튜닝 클래스 라이브러리 불러들이기

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
### 1. 랜덤포래스트 모델 클래스 생성하기
# - 앙상블 모델은 랜덤 규칙을 지정해서 생성함
rf_reg = RandomForestRegressor(random_state=42)
rf_reg


In [ ]:

### 2. 하이퍼파라메터 클래스를 사용하기 위해서
#     - 해당 훈련 모델에서 사용하는 하이퍼파라메터 정의하기
#       -> 딕셔너리 타입으로 정의해야함
#       -> 딕셔너리의 value값은 리스트 타입으로 하이퍼파라메터에서 사용할 값들을 정의해야함

# 하이퍼파라메터의 값들을 저장할 딕셔너리 변수 정의
gridParams = {}

# - (하이퍼파라메터)트리 모델의 갯수 정의
gridParams["n_estimators"] = [50, 100]

# - (하이퍼파라메터)트리가 뻗어 나가는 최대깊이 정의
# None : 제한을 두지 않겠다는 의미
gridParams["max_depth"] = [None, 3, 10]
print(gridParams)

In [ ]:
### 하이퍼파라메터 튜닝 클래스(GridSearchCV)의 속성값 정의하기
# 평가방법 정의
#  - 튜닝시 성능 검증에 사용할 평가 방법으로 사용하게끔 정의
#  - 회귀모델과 분류모델은 사용하는 평가 이름이 다름
scoring = ["neg_mean_squared_error", "r2"]

# 모델 선정 기준 정의
#  - 여러번 모델 수행한 결과 중에 가장 좋은 시점을 선정하는 방법으로 r2값을 기준으로 선정하게끔 정의
refit = "r2"

# 교차검증에 사용할 Fold(내부적으로 훈련데이터를 쪼개어 사용)의 갯수 정의
#  - 주로 3 또는 5를 사용(홀수로 사용), 성능에 많은 영향 미침
cv = 5

# CPU Core 갯수 정의
#  -1은 시스템에 있는 모든 core의 갯수를 사용하겠다는 의미(보통 전체 사용)
n_jobs = -1

In [ ]:
### 하이퍼파라메터 튜닝 클래스 생성하기 = 훈련 모델과 동일함
grid_search_model = GridSearchCV(
    # 하이퍼파라메터 튜닝에 사용할 원본(실체) 모델을 넣기(깨끗한 상태의 모델)
    estimator = rf_reg,

    # 사용할 하이퍼파라메터 딕셔너리 데이터 넣어주기
    param_grid = gridParams,

    # 평가 방법 넣어주기(자체적으로 튜닝시에 성능을 비교할 평가 방법)
    scoring = scoring,

    # 선정 기준 넣어주기(자체적으로 튜닝 시 좋은 시점을 선정하는 기준)
    refit = refit,

    # 교차검증에 사용할 Fold 갯수 정의
    cv = cv,

    # CPU Core 갯수 넣어주기(시스템 자원을 얼만큼 사용할 것이지)
    n_jobs = n_jobs
)

grid_search_model

In [ ]:
### 하이퍼파라메터 튜닝 클래스 모델 훈련(학습) 시키기
# - 기존 훈련 모델 훈련 순서와 동일
# - 넣어준 하이퍼파라메터들을 이용해서 가장 좋은 시점의 성능을 가지는 값을 찾아줌
# - 가장 좋은 시점의 하이퍼파라메터 값들로 [훈련(학습)이 완료된 모델]을 정해 놓고 있음
grid_search_model.fit(X_train_scaled, y_train)

### 튜닝 결과 추출하기

In [ ]:
### 최적의 하이퍼파라메터 추출
best_params = grid_search_model.best_params_
print(best_params)

### 최적의 성능(결정계수) 추출하기
best_score = grid_search_model.best_score_
print(best_score)

### [최적의 하이퍼파라메터가 적용된 모델] 추출하기
best_model = grid_search_model.best_estimator_
best_model

In [ ]:
"""
<최적의 모델을 이용해서 아래 수행>
 1. 검증데이터 및 테스트 데이터로 [예측]
 2. 검증데이터 및 테스트 데이터로 [오차 및 R2-score] 확인 
    : 검증 R2-score값을 훈련 score와 비교하여 과적합 여부 확인
 3. 다른 모델도 튜닝 클래스에 적용 후 1번~2번 확인
 4. 최종 모델 선정
"""

In [ ]:
### 1. 검증데이터 및 테스트 데이터로 [예측]
val_pred  = best_model.predict(X_val_scaled)
test_pred = best_model.predict(X_test_scaled)

### 2. 검증데이터 및 테스트 데이터로 [오차 및 R2-score] 확인 
#    : 검증 R2-score값을 훈련 score와 비교하여 과적합 여부 확인
val_mae  = mean_absolute_error(y_val,  val_pred)
test_mae = mean_absolute_error(y_test, test_pred)

val_r2  = r2_score(y_val,  val_pred)
test_r2 = r2_score(y_test, test_pred)

print(f"{best_score:.4f}, {val_r2:.4f}, {test_r2:.4f}, 과적합여부:{best_score - val_r2:.4f}")

### 전체 모델에 대해서 하이퍼파라메터 튜닝 클래스 적용하기

In [ ]:
"""
<기본적으로 모델별 사용가능한 하이퍼파라메터>
 1. LinearRegression, Ridge, Lasso
  - 선형방정식에서 절편사용여부(기본값 True) : fit_intercept = [True, False]
  - 예측값을 양수로 사용할지 여부(기본값 True) : positive = [True, False]
 
 2. GradentBoostingRegressor
  - 트리 갯수(보통 50~1000 사이값 사용) : n_estimators = [50, 100]
  - 트리 깊이(보통 3~10 사이값 사용) : max_depth = [None, 3, 10]
  - 학습률(보폭)(보통 0.01~0.2 사이값 사용) : learning_rate = [0.01, 0.1, 0.2]
 
 3. HistGradientBoostingRegressor
  - 트리 갯수(보통 50~1000 사이값 사용) : max_iter = [50, 100]
  - 트리 깊이(보통 3~10 사이값 사용) : max_depth = [None, 3, 10]
  - 학습률(보폭)(보통 0.01~0.2 사이값 사용) : learning_rate = [0.01, 0.1, 0.2]
 
 4. RandomForestRegressor, XGBRegressor
  - 트리 갯수(보통 50~1000 사이값 사용) : n_estimators = [50, 100]
  - 트리 깊이(보통 3~10 사이값 사용) : max_depth = [None, 3, 10]
"""

In [ ]:
"""
 1. 7개 모델 반복 수행
 2. 반복 중에 모델별로 사용할 하이퍼파라메터 선택해야함
    (조건문 사용해야함 또는 각자의 방법 적용)
 3. 튜닝 클래스 훈련 시키기
 4. 최적의 결정계수(훈련결정계수)와 모델 추출
 5. 훈련과 검증 데이터로 예측 및 결정계수 확인
 6. 과적합 여부 확인

 => 반복 시 출력값 : 훈련결정계수, 검증결정계수, 과적합여부

 7. 출력결과를 이용해서 최종 모델 선정
 8. 최종 선정 모델을 사용할 수 있도록 프로그램 처리
"""

In [ ]:
### 7개 훈련모델 생성하기
liner_reg = LinearRegression()
ridge_reg = Ridge()
lasso_reg = Lasso()
rf_reg    = RandomForestRegressor(random_state=42)
gb_reg    = GradientBoostingRegressor(random_state=42)
hgb_reg   = HistGradientBoostingRegressor(random_state=42)
xgb_reg   = XGBRegressor(random_state=42)

best_models = {}
### 모델 훈련에 사용할 모든 모델을 리스트에 담기(반복 사용을 위해)
models = [liner_reg, ridge_reg, lasso_reg,
          rf_reg, gb_reg, hgb_reg, xgb_reg]


### GridSearchCV 모델 속성 정의하기
# - 평가방법(MSE, R2) 정의하기
scoring = ["neg_mean_squared_error", "r2"]


# 모델 선정 기준(비교대상) 정의하기
# - 위에 제시한 평가방법 중에 한개 정의
refit = "r2"


# 교차검증에 사용할 Fold(쪼갤 그룹)의 갯수 지정
# - 보통 사용하는 Fold 갯수 : 3 또는 5 (더 큰값 사용가능, 홀수로 사용)
cv = 5


# CPU Core(코어) 사용 갯수 정의하기
# -1은 : 모든 코어를 다 사용하겠다는 의미
n_jobs = -1
   
for m in models :
    ### 클래스 이름 추출 : 이름은 문자열 입니다.
    m_nm = m.__class__.__name__
    print(f"----------{m_nm}----------")
   
    ###  하이퍼파라메터를 저장할 딕셔너리 변수 정의
    # - 반복 할 때마다 깨끗하게 만들기
    gridParams = {}
   
    if m_nm in ["LinearRegression", "Ridge", "Lasso"] :
        gridParams["fit_intercept"] = [True, False]
        gridParams["positive"]      = [True, False]
   
    elif m_nm in ["GradentBoostingRegressor"] :
        gridParams["n_estimators"]  = [50, 100]
        gridParams["max_depth"]     = [None, 3, 10]
        gridParams["learning_rate"] = [0.01, 0.1, 0.2]
   
    elif m_nm in ["HistGradientBoostingRegressor"] :
        gridParams["max_iter"] = [50, 100]
        gridParams["max_depth"] = [None, 3, 10]
        gridParams["learning_rate"] = [0.01, 0.1, 0.2]
   
    else :
        # RandomForestRegressor, XGBRegressor
        gridParams["n_estimators"] = [50, 100]
        gridParams["max_depth"] = [None, 3, 10]
   
    ### 하이퍼파라메터 튜닝 클래스 생성하기    
    grid_search_model = GridSearchCV(
        # 생성한 모델 넣기
        m,
        # 정의한 하이퍼파라메터 넣기(딕셔너리 타입)
        gridParams,
        # 평가방법(MSE, R2) 정의하기
        scoring = scoring,
        # 모델 선정 기준(비교대상) 정의하기
        refit = refit,
        # 교차검증에 사용할 Fold(쪼갤 그룹)의 갯수 지정
        cv = cv,
        # CPU Core(코어) 사용 갯수 정의하기
        n_jobs = n_jobs
    )
   
    ### 모델 훈련시키기
    grid_search_model.fit(X_train_scaled, y_train)
   
    ### 최적의 파라메터 및 모델 추출하기
    best_param = grid_search_model.best_params_
    best_model = grid_search_model.best_estimator_
    best_models[m_nm] = best_model

   
    ### 훈련 예측 및 평가하기
    train_pred = best_model.predict(X_train_scaled)
    train_mae = mean_absolute_error(y_train, train_pred)
    train_mse = mean_squared_error(y_train, train_pred)
    train_r2  = r2_score(y_train, train_pred)
   
    ### 검증 예측 및 평가하기
    val_pred = best_model.predict(X_val_scaled)
    val_mae = mean_absolute_error(y_val, val_pred)
    val_mse = mean_squared_error(y_val, val_pred)
    val_r2  = r2_score(y_val, val_pred)
   
    ### 과적합 여부 확인
    train_val = train_r2 - val_r2
   
    ### 테스트 예측 및 평가하기
    test_pred = best_model.predict(X_test_scaled)
    test_mae = mean_absolute_error(y_test, test_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    test_r2  = r2_score(y_test, test_pred)
   
    msg = f"train_mae={train_mae:.0f}, train_mse={train_mse:.0f}, train_r2={train_r2:.4f} / "
    msg += f"\nval_mae={val_mae:.0f}, val_mse={val_mse:.0f}, val_r2={val_r2:.4f} / "
    msg += f"test_mae={test_mae:.0f}, test_mse={test_mse:.0f}, test_r2={test_r2:.4f} / "
    msg += f"과적합여부={train_val:.4f} / best_param={best_param}"
   
    print(msg, "\n")




In [ ]:
hgb_model = best_models["HistGradientBoostingRegressor"]
hgb_model

In [ ]:
"""
(해석)
 - 사용된 7개 모델들을 비교한 결과,
 - 일반화가 잘되고, 정확도(결정계수)가 높으며, 오차가 낮은 모델로 HGB 모델을 선정
 - 선정된 HGB 모델을 파일로 저장하여 다른곳에서 사용(서비스 등등)할 수 있도록 함
"""

In [ ]:
"""
<저장해야 하는 객체들>
- 데이터 스케일링 객체
- 특성 추가객체
- 선정 모델 객체

<클래스 저장/불러들이는 방법>
1. joblib 방식
    - 사용 라이브러리 : import joblib
    - 주로 사용하는 파일 확장자 : .pkl
2. pickle 방식
    - 사용 라이브러리 : import pickle
    - 주로 사용하는 파일 확장자 : .pickle
"""

In [ ]:
# joblib 방식 모델 저장 및 불러들이기
import joblib

### 모델 저장시키기
# - 저장위치
save_path = "../models/hgb_model.pkl"

# - 저장하기
joblib.dump(hgb_model, save_path)

# - 파일 읽어들이기
file_path = "../models/hgb_model.pkl"
load_hgb_model = joblib.load(file_path)
load_hgb_model


In [ ]:
# pickle 방식 모델 저장 및 불러들이기
import pickle

# - with문 : 지속적 반복 또는 시간이 소요되는 처리를 수행할 때 주로 사용됨
#        : 처리가 완료 될때까지 다른 작업 못함(메모리 최소화 사용)
# - open() 파일 처리 함수
# - "wb" : w는 쓰기, b는 바이너리(파일에 바이너리 형태로 쓰기 작업을 수행)
# - "rb" : r는 읽기, b는 바이너리(파일에 바이너리 형태로 쓰기 작업을 수행)
# - as fw : as는 별칭 open된 파일을 fw 별칭 변수가 소유하고 있다는 의미(변수와 같음)

### 모델 저장시키기
# - 저장위치
save_path = "../models/hgb_model.pickle"

# - 저장하기
with open(save_path,"wb") as fw :
    pickle.dump(hgb_model, fw)

# - 파일 읽어들이기
file_path = save_path
with open(file_path, "rb") as f :
    load_hgb_model = pickle.load(f)

load_hgb_model

In [ ]:
### 위에서 데이터 처리에 사용된 클래스 중에
# - 데이터스케일링 클래스도 저장해서 관리해야함
# - 데이터스케일링 클래스 저장 시에는 fit()까지 완료된 변수를 이용하여 저장
# - 불러들인 데이터스케일 클래스는 transform() 부터 진행 (fit()은 절대 하면 안됨)

"""
<임의 값으로 예측해보기>
    - 면적 : 69.04
    - 층수 : 27층
    - 역까지 도보시간 : 21.4
    - 편의시설 점수 : 2.7
** 위 값으로 주택가격을 예측해 주세요
    - 순서 : 2차원 독립변수로 -> 데이터스케일링 변환만 시키기(이미 fit은 되어있기 때문에) -> 예측진행
"""

In [ ]:
home_data = [[69.04, 27, 21.4, 2.7]]

home_data_scaled = ss.transform(home_data)

pred = load_hgb_model.predict(home_data_scaled)

print("예측 주택가격 :", pred[0])